# Phase 0: challenger smoke test (Colab T4)

Checks that each challenger model downloads, loads and returns output on **one demo image**. This is not the benchmark.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

Takes about 30–45 min, mostly downloads (Qwen3-VL-8B alone is ~17 GB). At the end a zip of the results downloads; send it back.

Models: Qwen3-VL-8B (4-bit), GLM-OCR, MinerU 2.5 Pro, Typhoon OCR 1.5, Uni-MuMER (3B), DECIMER (hand-drawn). No hosted APIs are called; every model runs on this GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

In [ ]:
# Get the benchmark code and demo images (public repo)
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/VickyKumarOfficial/pipeline-ocr_model.git /content/repo
%cd /content/repo
!git log --oneline -1

In [ ]:
# Torch models. mineru-vl-utils first, then transformers last so nothing downgrades it
# (GLM-OCR's card asks for transformers >= 5.3).
!pip install -q mineru-vl-utils qwen-vl-utils accelerate bitsandbytes
!pip install -q -U "transformers>=5.3"
!pip list 2>/dev/null | grep -iE "^(torch|transformers|accelerate|bitsandbytes|mineru-vl-utils) " 

In [ ]:
# Each model runs in its own process; results go to benchmark/results/smoke_colab/
!python benchmark/harness/smoke_colab.py

In [ ]:
# DECIMER needs TensorFlow + Python 3.10, so it gets its own environment
!pip install -q uv
!uv venv -q /content/decimer-env --python 3.10
!uv pip install -q -p /content/decimer-env/bin/python decimer
!/content/decimer-env/bin/python benchmark/harness/smoke_colab.py --child decimer | tee /content/decimer_result.txt

In [ ]:
import json, pathlib, shutil, subprocess
out = pathlib.Path("benchmark/results/smoke_colab")
summary = json.loads((out / "summary.json").read_text())
line = next((l for l in open("/content/decimer_result.txt") if l.startswith("RESULT ")), None)
summary = [s for s in summary if s["model_id"] != "decimer"] + [json.loads(line[7:]) if line else {"model_id": "decimer", "ok": False}]
env = {"gpu": subprocess.getoutput("nvidia-smi --query-gpu=name,driver_version --format=csv,noheader"),
       "pip": subprocess.getoutput("pip list 2>/dev/null | grep -iE '^(torch|transformers|accelerate|bitsandbytes|mineru-vl-utils) '"),
       "decimer_pip": subprocess.getoutput("/content/decimer-env/bin/python -m pip list 2>/dev/null | grep -iE '^(decimer|tensorflow) ' || uv pip list -p /content/decimer-env/bin/python | grep -iE '^(decimer|tensorflow) '")}
(out / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
(out / "environment.json").write_text(json.dumps(env, indent=2))
for s in summary:
    print(f"{s['model_id']:22} {'OK ' if s.get('ok') else 'FAIL'} {s.get('total_s_incl_load','')}s  gpu {s.get('peak_gpu_mb','')} MB")
    print("   ", (s.get("output_preview") or s.get("error", ""))[:300].replace("\n", " | "))

In [ ]:
shutil.make_archive("/content/smoke_colab", "zip", out)
from google.colab import files
files.download("/content/smoke_colab.zip")